In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
df = pd.read_csv('creditcard.csv')

print(df.head())
print(df.shape)
print(df.describe())
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nDataset Shape:", df.shape)
print("\nDataset Information:")
print(df.info())
features = ['Avg_Credit_Limit', 'Total_Credit_Cards',
            'Total_visits_bank', 'Total_visits_online',
            'Total_calls_made']

df[features].hist(figsize=(12, 8), bins=20)
plt.tight_layout()
plt.show()
X = df[features]

scaler = StandardScaler()
X = scaler.fit_transform(X).astype('float32')

print("Processed Data Shape:", X.shape)
x_train, x_test = train_test_split(
    X, test_size=0.2, random_state=42
)

print("Training Data:", x_train.shape)
print("Testing Data:", x_test.shape)
input_dim = x_train.shape[1]

input_layer = Input(shape=(input_dim,))

encoded = Dense(4, activation='relu')(input_layer)
encoded = Dense(2, activation='relu')(encoded)

decoded = Dense(4, activation='relu')(encoded)
decoded = Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(input_layer, decoded)
autoencoder.summary()
autoencoder.compile(optimizer='adam', loss='mse')
history = autoencoder.fit(
    x_train, x_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    shuffle=True
)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.title('Autoencoder Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()
predictions = autoencoder.predict(x_test)

mse = np.mean(np.square(x_test - predictions), axis=1)

print("Reconstruction Errors:")
print(mse[:10])
threshold = np.percentile(mse, 95)

print("Anomaly Threshold:", threshold)

predictions = (mse > threshold).astype(int)
results = pd.DataFrame({
    'Actual_Index': x_test.shape[0] and np.arange(len(x_test)),
    'Reconstruction_Error': mse,
    'Prediction': predictions
})

results['Prediction'] = results['Prediction'].map({
    0: 'Normal',
    1: 'Anomaly'
})

print(results.head(20))
print("\nAnomaly Counts:")
print(results['Prediction'].value_counts())
plt.figure(figsize=(10, 5))

plt.scatter(range(len(mse)), mse, label='Reconstruction Error')
plt.axhline(threshold, color='red', linestyle='--', label='Threshold')

plt.title('Customer Anomaly Detection')
plt.xlabel('Customer Index')
plt.ylabel('Reconstruction Error')
plt.legend()
plt.show()